# Análise do Sistema de Informação sobre Mortalidade – SIM

### 1. Origem dos dados, suas características e recorte temporal.

Os dados foram obtidos por meio do **Portal Brasileiro de Dados Abertos do Governo Federal**. A base é desenvolvida, consolidada e coordenada pelo **Ministério da Saúde**.
O arquivo **Mortalidade_Geral_2026.csv** consiste em um conjunto de dados públicos estruturado que reúne os registros oficiais de óbitos em território nacional. Ele possui **86 colunas**, cujas descrições e características constam no dicionário de dados **(Dicionario_SIM_2025.pdf)**, também extraído do **Portal Brasileiro de Dados Abertos do Governo Federal**.
A referida base de dados foi delimitada ao **período de 01/01/2026 a 31/05/2026**, permitindo uma análise focada no desempenho do primeiro semestre do ano de 2026, totalizando **506531 linhas**.

### Perguntas a serem respondidas:

1. Quantidade de mortes por mês?
2. Quantidade de mortes por estado?
3. Quantidade de mortes por faixa etária?
4. As 5 maiores causas de morte(CID)?
5. As 5 cidades com maior número de óbitos? 

### 2. Diagnóstico de qualidade

2.1 - Dimensões, tipos, uso de memória.

In [41]:
import pandas as pd
import numpy as np

mortalidade_completa = pd.read_csv(
    '../dados/Mortalidade_Geral_2026.csv',
    sep=';',
    parse_dates=['DTOBITO', 'DTNASC'],
    date_format='%d/%m/%Y',
    dtype={'CAUSAMAT': str}
)

# Dimensões
print("Dimensões (Linhas, Colunas):", mortalidade_completa.shape)
# Tipos
print("\nTipos (Colunas, Tipos):")
print(mortalidade_completa.dtypes)
# Uso de Memória
print("\nMemória Utilizada:")
mortalidade_completa.info(verbose=False, memory_usage='deep')


Dimensões (Linhas, Colunas): (506531, 86)

Tipos (Colunas, Tipos):
contador        int64
ORIGEM          int64
TIPOBITO        int64
DTOBITO           str
HORAOBITO     float64
               ...   
ALTCAUSA      float64
CAUSABAS_O        str
TPPOS             str
TP_ALTERA     float64
CB_ALT            str
Length: 86, dtype: object

Memória Utilizada:
<class 'pandas.DataFrame'>
RangeIndex: 506531 entries, 0 to 506530
Columns: 86 entries, contador to CB_ALT
dtypes: float64(56), int64(12), str(18)
memory usage: 667.1 MB


2.2 - Faltantes por coluna (quantidade e percentual).

In [42]:
# Quantidade absoluta de dados faltantes por coluna
print("--- Quantidade de Faltantes ---")
print(mortalidade_completa.isnull().sum())

# Percentual de dados faltantes por coluna (arredondado em 2 casas decimais)
print("\n--- Percentual (%) de Faltantes ---")
print((mortalidade_completa.isnull().mean() * 100).round(2))


--- Quantidade de Faltantes ---
contador           0
ORIGEM             0
TIPOBITO           0
DTOBITO            0
HORAOBITO      14739
               ...  
ALTCAUSA      502188
CAUSABAS_O       990
TPPOS         173848
TP_ALTERA     496208
CB_ALT        504104
Length: 86, dtype: int64

--- Percentual (%) de Faltantes ---
contador       0.00
ORIGEM         0.00
TIPOBITO       0.00
DTOBITO        0.00
HORAOBITO      2.91
              ...  
ALTCAUSA      99.14
CAUSABAS_O     0.20
TPPOS         34.32
TP_ALTERA     97.96
CB_ALT        99.52
Length: 86, dtype: float64


2.3 - Duplicados, categorias inconsistentes, valores inválidos.

In [43]:
# Quantidade total de linhas 100% duplicadas na tabela inteira
total_duplicados = mortalidade_completa.duplicated().sum()
print(f"Total de linhas duplicadas: {total_duplicados}")
mortalidade_completa[mortalidade_completa.duplicated(keep=False)]

# Categorias Inconsistentes
print("\nColuna RACACOR:")
mortalidade_completa['RACACOR'].value_counts(dropna=False)

Total de linhas duplicadas: 0

Coluna RACACOR:


RACACOR
1.0    250465
4.0    198952
2.0     45706
NaN      6245
3.0      3246
5.0      1917
Name: count, dtype: int64

2.4 - Identificação de outliers com método justificado (IQR ou z-score), avaliados no **grupo de comparação correto**

In [44]:
# Resumo estatístico das colunas numéricas (atenção para os valores 'min' e 'max')
mortalidade_completa.describe()

,contador,ORIGEM,TIPOBITO,HORAOBITO,NATURAL,CODMUNNATU,IDADE,SEXO,RACACOR,ESTCIV,...,NUDIASOBCO,DTCADINV,TPOBITOCOR,DTCONINV,TPRESGINFO,DTCADINF,MORTEPARTO,DTCONCASO,ALTCAUSA,TP_ALTERA
count,506531.000000,506531.0,506531.0,491792.000000,491086.000000,485169.000000,506531.000000,506531.000000,500286.000000,486504.000000,...,12631.000000,1.273000e+04,12730.000000,1.262500e+04,478.000000,4.565000e+03,4565.000000,4.087000e+03,4343.000000,10323.000000
mean,253266.000000,1.0,2.0,1201.072453,825.196057,317515.523273,463.641740,1.456485,2.312693,2.522725,...,43.725754,1.680468e+07,8.840613,1.642395e+07,1.305439,1.736258e+07,3.298357,1.689020e+07,1.825927,14.785624
std,146223.048939,0.0,0.0,669.959742,67.062695,83993.698090,46.286529,0.498737,1.426377,1.699038,...,30.848502,8.546333e+06,0.848469,8.530842e+06,0.567103,8.420059e+06,1.351602,8.527233e+06,0.379216,4.822529
min,1.000000,1.0,2.0,0.000000,3.000000,110000.000000,1.000000,0.000000,1.000000,1.000000,...,0.000000,1.032026e+06,1.000000,1.012026e+06,1.000000,1.042026e+06,1.000000,1.042026e+06,1.000000,2.000000
25%,126633.500000,1.0,2.0,635.000000,826.000000,260790.000000,457.000000,1.000000,1.000000,1.000000,...,20.000000,9.042026e+06,9.000000,9.032026e+06,1.000000,1.004203e+07,3.000000,9.042026e+06,2.000000,17.000000
50%,253266.000000,1.0,2.0,1205.000000,831.000000,314800.000000,471.000000,1.000000,1.000000,2.000000,...,37.000000,1.704203e+07,9.000000,1.604203e+07,1.000000,1.803203e+07,3.000000,1.703203e+07,2.000000,17.000000
75%,379898.500000,1.0,2.0,1757.000000,835.000000,355030.000000,482.000000,2.000000,4.000000,3.000000,...,62.000000,2.502203e+07,9.000000,2.403203e+07,1.750000,2.505203e+07,3.000000,2.502203e+07,2.000000,17.000000
max,506531.000000,1.0,2.0,2359.000000,999.000000,530010.000000,999.000000,2.000000,5.000000,9.000000,...,147.000000,3.105203e+07,9.000000,3.105203e+07,3.000000,3.105203e+07,9.000000,3.103203e+07,2.000000,17.000000


In [45]:
# 1. Define a lista de colunas que serão importadas
colunas_desejadas = [
    'contador',
    'TIPOBITO',
    'DTOBITO',
    'DTNASC',
    'IDADE',
    'SEXO',
    'RACACOR',
    'ESTCIV',
    'ESC2010',
    'OCUP',
    'CODMUNRES',
    'CODMUNOCOR',
    'CAUSABAS'
]

# 2. Defina os tipos de dados das colunas
tipos_dados = {
    'contador': int,
    'TIPOBITO': str,
    'IDADE': str,       # Mantém como texto para não perder os zeros à esquerda da codificação do DATASUS
    'SEXO': str,
    'RACACOR': str,
    'ESTCIV': str,
    'ESC2010': str,
    'OCUP': str,
    'CODMUNRES': str,
    'CODMUNOCOR': str,
    'CAUSABAS': str     # Garante que códigos CID mistos não gerem alertas
}

# 3. Importa o arquivo CSV de forma otimizada
mortalidade = pd.read_csv(
    'Mortalidade_Geral_2026.csv',
    sep=';',
    usecols=colunas_desejadas,
    dtype=tipos_dados,
    parse_dates=['DTOBITO', 'DTNASC'],
    date_format='%d/%m/%Y'
)

# 4. Converte DTOBITO e DTNASC (formato: ddmmYYYY) para uma data real do Pandas (formato: YYYY-mm-dd)
mortalidade['DTOBITO'] = pd.to_datetime(mortalidade['DTOBITO'], format='%d%m%Y', errors='coerce')
mortalidade['DTNASC'] = pd.to_datetime(mortalidade['DTNASC'], format='%d%m%Y', errors='coerce')


FileNotFoundError: [Errno 2] No such file or directory: 'Mortalidade_Geral_2026.csv'

In [ ]:
# Mostra o tipo de dado das colunas convertidas em data real (formato: YYYY-mm-dd)
print(mortalidade[['DTOBITO', 'DTNASC']].dtypes)

# Mostra o formato das datas convertidas (YYYY-mm-dd)
mortalidade.head()

In [ ]:
mortalidade.info()

In [ ]:
# 1. Separar a IDADE em Tipo (1º dígito) e Quantidade (2º e 3º dígitos)
mortalidade['IDADELIMPA'] = ("000" + mortalidade['IDADE'].astype(str)).str[-3:]

tipo_idade = mortalidade['IDADELIMPA'].str[0]
qtde_idade = pd.to_numeric(mortalidade['IDADELIMPA'].str[1:], errors='coerce')

qtde_idade.loc[qtde_idade == 0] = 1

# 2. Aplicar a regra de conversão para ANOS
mortalidade['IDADEANOS'] = np.select(
    [
        tipo_idade == '0',  # Minutos
        tipo_idade == '1',  # Horas
        tipo_idade == '2',  # Dias
        tipo_idade == '3',  # Meses
        tipo_idade == '4',  # Anos (< 100)
        tipo_idade == '5'   # Anos (>= 100)
    ],
    [
        qtde_idade / (60 * 24 * 365),
        qtde_idade / (24 * 365),
        qtde_idade / 365,
        qtde_idade / 12,
        qtde_idade,
        qtde_idade + 100
    ],
    default=np.nan  # Idades ignoradas ou nulas
)

# 3. Remove a coluna temporária auxiliar
# mortalidade.drop(columns=['IDADELIMPA'], inplace=True)

mortalidade.head()

In [ ]:
# Usando o método .query() (Sintaxe mais limpa)
# mortalidade.query("IDADEANOS < 0.01")

# mortalidade[mortalidade['IDADEANOS'] <= 0.00274] # Idade < que 1 dia
mortalidade[mortalidade['IDADEANOS'] > 0.00274] # Idade > que 1 dia
# mortalidade[(mortalidade['IDADELIMPA'] == '200') & (mortalidade['CODMUNRES'] == '351360')]
# mortalidade[mortalidade['IDADELIMPA'] == '201']
# mortalidade[mortalidade['CODMUNRES'] == '351360']

In [ ]:
mortalidade.describe().round(7)

In [ ]:
mortalidade.info()

In [ ]:
# 1. Define a lista de colunas que serão importadas
colunas_desejadas = [
    'CODMUNRES',
    'MUNICIPIO',
    'UF',
    'POPULAÇÃO'
]

# 2. Defina os tipos de dados das colunas
tipos_dados = {
    'CODMUNRES': str,
    'MUNICIPIO': str,
    'UF': str,
    'POPULAÇÃO': str
}

# 3. Importa o arquivo CSV de forma otimizada
municipios = pd.read_excel(
    'MUNICIPIOS.xlsx',
    usecols=colunas_desejadas,
    dtype=tipos_dados,
    na_values=['null']
)

In [ ]:
# Substitui os nulos da coluna por 0 e converte para inteiro comum
municipios['POPULAÇÃO'] = municipios['POPULAÇÃO'].fillna(0).astype('int64')

# Renomeando a coluna
municipios = municipios.rename(columns={'CODMUNRES': 'CODMUNOCOR'})

municipios.info()

In [ ]:
mort_muni = pd.merge(mortalidade, municipios, on='CODMUNOCOR', how='left')

mort_muni.head()